# SocialChoice 02 : Mariage Stable — Algorithme de Gale-Shapley (twin C# .NET)

> **Twin C# .NET de `GameTheory/SocialChoice/02-Stable-Marriage-Gale-Shapley.ipynb`** (binôme identique SC-01..04, issue #19681). Kernel `.net-csharp`. Ce carnet rejoue le même arc pédagogique que le jumeau Python — profil Knuth (1976) n=4 à l'identique, trace pas-à-pas, vérification de stabilité, man-optimalité par énumération exhaustive — implémenté from-scratch en C# .NET 9 (BCL pure : `System`, `System.Collections.Generic`, `System.Linq`, 0 NuGet). Les **preuves** vivent dans le lake Lean `game_theory_lean/StableMarriage/` : le C# **exécute** l'algorithme, le lake le **prouve** — ce carnet n'en réimplémente aucune.

**Navigation** : [<< 01b-Lean-SocialChoice-Formal.ipynb](01b-Lean-SocialChoice-Formal.ipynb) | [03-Voting-Methods-Csharp.ipynb >>](03-Voting-Methods-Csharp.ipynb) | [Index](README.md)

**Kernel** : .NET (C#)

***

## Introduction

Le **problème du mariage stable** (Gale & Shapley, 1962) modélise une situation bilatérale : `n` hommes et `n` femmes expriment leurs préférences sur l'ensemble opposé, et on cherche un **couplage parfait** (bijection) tel qu'aucune paire (homme, femme) ne se préfère mutuellement à ses partenaires assignés. C'est l'archétype des problèmes d'**affectation avec stabilité** : admission universitaire (étudiants ↔ départements), résidents ↔ hôpitaux, plateformes bilatérales (chauffeurs ↔ passagers).

L'algorithme de **Gale-Shapley** (acceptation différée, "deferred acceptance") résout le problème en `O(n²)` propositions, et la théorie montre qu'il est **man-optimal** : chaque homme obtient sa meilleure partenaire possible parmi tous les couplages stables, et dualement chaque femme sa pire. C'est l'un des rares algorithmes d'allocation qui offre une garantie aussi forte avec un protocole si simple.

### Idée clé

Ce notebook implémente l'algorithme en C#, trace son exécution pas-à-pas sur un exemple `n=4`, puis vérifie empiriquement les deux garanties fondamentales (stabilité et optimalité homme) en les comparant à une **énumération exhaustive** des couplages stables. Il est le **twin C#** du jumeau Python `02-Stable-Marriage-Gale-Shapley.ipynb` et le compagnon exécutable du lake formel `game_theory_lean/StableMarriage/` (aucun `sorry` sur les théorèmes principaux) : la même structure, prouvée cette fois sans appel à la machine Lean.

### Objectifs d'apprentissage

1. Comprendre l'algorithme d'acceptation différée et sa terminaison en `n²` propositions
2. Savoir détecter une **paire bloquante** et prouver qu'un couplage est **stable**
3. Vérifier empiriquement la **man-optimalité** par énumération exhaustive des couplages stables
4. Faire le pont avec la formalisation Lean 4 du même algorithme

### Prérequis

- Notions élémentaires de théorie des ensembles (bijection, fonction injective)
- Familiarité avec C# (classes, `List<T>`, `Dictionary<TKey, TValue>`, LINQ)
- Idéalement : avoir lu SC-01 (Arrow) ou SC-03 (méthodes de vote) pour le contexte choix social

### Durée estimée : 35 minutes

> *Ancres savantes — Gale, D. & Shapley, L.S. (1962), College Admissions and the Stability of Marriage, **The American Mathematical Monthly** 69(1):9-15 (premier article posant le problème et prouvant la convergence en au plus `n²` propositions ; prix Nobel d'économie 2012 à Lloyd Shapley pour ses contributions à la théorie des jeux et de l'allocation, dont ce résultat fondateur) ; Knuth, D.E. (1976), **Mariages Stables et leurs relations avec d'autres problèmes combinatoires**, Les Presses de l'Université de Montréal (introduction canonique du problème en langue française ; traduction anglaise révisée : *Stable Marriage and Its Relation to Other Combinatorial Problems*, CRM Proceedings & Lecture Notes, AMS 1997) ; Roth, A.E. (2008), Deferred Acceptance Algorithms: History, Theory, Practice, and Open Questions, **International Journal of Game Theory** 36(3-4):537-569 (récit historique de l'application au National Resident Matching Program, 1952-depuis, et reformulation théorique moderne) ; Gusfield, D. (1989), Three still open problems in the theory of stable marriage, Technical Report CS-89-21, UC Davis (questions structurelles sur la structure du treillis des mariages stables, soubassement du lake Lean `Lattice.lean`).*

## 0. Configuration

Usings (BCL pure, 0 NuGet) et graine pour reproductibilité — miroir de la cellule `random.seed(42)` du jumeau Python.

In [1]:
// SocialChoice 02 : Mariage Stable -- Algorithme de Gale-Shapley -- twin C# de 02-Stable-Marriage-Gale-Shapley
// (issue #19681, binome identique SC-01..04). Le jumeau Python est le carnet miroir ; les preuves
// formelles vivent dans game_theory_lean/StableMarriage/ (gsGaleShapley, IsStable, IsManOptimal).
// BCL pure : System / System.Collections.Generic / System.Linq -- 0 NuGet.
using System;
using System.Collections.Generic;
using System.Linq;

// Graine pour reproductibilite (miroir de random.seed(42) du twin Python).
// Consommee par l'exercice 3 (profils aleatoires).
var rng = new Random(42);

Console.WriteLine("Configuration OK : SocialChoice 02 - Mariage Stable (Gale-Shapley, twin C# .NET 9)");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Configuration OK : SocialChoice 02 - Mariage Stable (Gale-Shapley, twin C# .NET 9)


## 1. Définitions : PrefProfile et Matching

On modélise `n` hommes et `n` femmes par des indices `0, 1, ..., n-1`. Chaque personne a un **classement strict** (permutation) de l'ensemble opposé.

**Convention** : un rang **plus petit** signifie une préférence **plus forte** (cohérent avec le lake Lean `PrefProfile` où `menPref` est bijectif et `manPrefers` compare des rangs).

In [2]:
// === Section 1 : PrefProfile -- preferences strictes des deux cotes ===
// Miroir C# de la classe PrefProfile du twin Python ; correspond a la structure Lean
// PrefProfile (Definitions.lean:36) ou menPref est bijectif. MenPref[m][k] = la femme
// de rang k pour l'homme m (rang 0 = preferee).

public class PrefProfile
{
    public readonly int N;
    public readonly List<List<int>> MenPref;
    public readonly List<List<int>> WomenPref;

    public PrefProfile(List<List<int>> menPref, List<List<int>> womenPref)
    {
        int n = menPref.Count;
        if (womenPref.Count != n)
            throw new ArgumentException("hommes et femmes de cardinalite differente");
        foreach (var p in menPref)
            if (!p.OrderBy(x => x).SequenceEqual(Enumerable.Range(0, n)))
                throw new ArgumentException($"preference homme non permutation : [{string.Join(", ", p)}]");
        foreach (var p in womenPref)
            if (!p.OrderBy(x => x).SequenceEqual(Enumerable.Range(0, n)))
                throw new ArgumentException($"preference femme non permutation : [{string.Join(", ", p)}]");
        N = n;
        MenPref = menPref;
        WomenPref = womenPref;
    }

    // Rang de la femme w dans la liste de l'homme m (0 = preferee).
    public int MenRank(int m, int w) => MenPref[m].IndexOf(w);

    // Rang de l'homme m dans la liste de la femme w (0 = prefere).
    public int WomenRank(int w, int m) => WomenPref[w].IndexOf(m);

    // L'homme m prefere strictement la femme w1 a w2 -- manPrefers (Definitions.lean:55).
    public bool ManPrefers(int m, int w1, int w2) => MenRank(m, w1) < MenRank(m, w2);

    // La femme w prefere strictement l'homme m1 a m2 -- womanPrefers (Definitions.lean:62).
    public bool WomanPrefers(int w, int m1, int m2) => WomenRank(w, m1) < WomenRank(w, m2);

    public override string ToString() => $"PrefProfile(n={N})";
}

// Petit test : l'identite 0->0, 1->1, 2->2 est coherente
var profTest = new PrefProfile(
    new() { new() { 0, 1, 2 }, new() { 0, 1, 2 }, new() { 0, 1, 2 } },
    new() { new() { 0, 1, 2 }, new() { 0, 1, 2 }, new() { 0, 1, 2 } });
Console.WriteLine(profTest);
Console.WriteLine($"m0 prefere w1 a w2 ? {profTest.ManPrefers(0, 1, 2)}");  // True
Console.WriteLine($"m0 prefere w0 a w1 ? {profTest.ManPrefers(0, 0, 1)}");  // True
Console.WriteLine($"m0 prefere w1 a w0 ? {profTest.ManPrefers(0, 1, 0)}");  // False

PrefProfile(n=3)


m0 prefere w1 a w2 ? True


m0 prefere w0 a w1 ? True


m0 prefere w1 a w0 ? False


### Interprétation : convention de rang

On note `MenRank(m, w)` le **rang** de la femme `w` dans la liste de l'homme `m` (0 = préférée). L'homme préfère `w1` à `w2` si et seulement si son rang est plus petit — c'est la convention `manPrefers` du lake `Definitions.lean` (ligne 55-56 : `prof.menPref m w1 < prof.menPref m w2`).

**Pourquoi cette convention plutôt qu'une liste d'objets préférés ?** Elle permet d'interroger le rang d'une personne donnée en un appel. Une liste `MenPref[m] = [w_preferee_1, w_preferee_2, ...]` est plus naturelle à lire, mais pour vérifier si `m` préfère `w1` à `w2`, il faut faire `MenPref[m].IndexOf(w1) < MenPref[m].IndexOf(w2)` en O(n) (l'équivalent du `.index()` du jumeau Python). Pour `n` jusqu'à 100, c'est acceptable ; pour `n > 1000`, on construirait un cache `rank[m][w] → int`.

Le lake Lean échappe à ce compromis : sa représentation **est** la fonction de rang (`menPref : Fin n → Fin n → Fin n` associe à chaque femme son rang), la comparaison est donc directe sans recalcul de position.

## 2. L'algorithme de Gale-Shapley (hommes proposent)

### Pseudo-code

1. Initialement, tous les hommes et toutes les femmes sont **libres**.
2. Tant qu'il existe un homme libre `m` qui n'a pas encore proposé à toutes les femmes :
   - `m` propose à la femme `w` qu'il préfère parmi celles à qui il n'a pas encore proposé.
   - Si `w` est libre : elle accepte `m`.
   - Si `w` est engagée à `m_old` : elle compare les deux. Si elle préfère `m` à `m_old`, elle accepte `m` et libère `m_old`. Sinon, elle rejette `m` (et `m` redevient libre).
3. L'algorithme termine en au plus `n²` propositions (chaque homme propose à chaque femme au plus une fois).

In [3]:
// === Section 2 : Gale-Shapley hommes-proposent, avec trace pas-a-pas ===
// Miroir C# de gale_shapley(prof, trace) du twin Python. Correspond a gsGaleShapley
// du lake (GSState.lean:137) : le couplage retourne satisfait IsStable et IsManOptimal
// (ces deux theoremes sont prouves dans GaleShapley.lean, lignes 104 et 130).

public sealed record GsResult(Dictionary<int, int> Matching, List<string> Log);

public static class Gs
{
    // Algorithme de Gale-Shapley, version hommes-proposent (acceptation differee).
    // Retourne le couplage (bijection hommes -> femmes) et le journal d'execution.
    public static GsResult GaleShapley(PrefProfile prof, bool trace = false)
    {
        int n = prof.N;
        var menMatch = new int?[n];    // homme -> femme engagee (null = libre)
        var womenMatch = new int?[n];  // femme -> homme engage
        var proposed = new bool[n, n]; // proposed[m, w] : m a-t-il deja propose a w ?
        var log = new List<string>();
        int step = 0;

        int? FreeManWithOptions()
        {
            for (int m = 0; m < n; m++)
            {
                if (menMatch[m] is null)
                {
                    for (int w = 0; w < n; w++)
                        if (!proposed[m, w]) return m;
                }
            }
            return null;
        }

        while (true)
        {
            var found = FreeManWithOptions();
            if (found is null)
            {
                if (trace) log.Add($"Step {step}: aucun homme libre restant -- FIN");
                break;
            }
            int m = found.Value;
            // w preferee non encore proposee par m
            int w = prof.MenPref[m].First(x => !proposed[m, x]);
            proposed[m, w] = true;
            step++;
            if (trace) log.Add($"Step {step}: homme {m} propose a femme {w}");
            var mCurrent = womenMatch[w];
            if (mCurrent is null)
            {
                womenMatch[w] = m;
                menMatch[m] = w;
                if (trace) log.Add($"  -> femme {w} accepte (libre)");
            }
            else if (prof.WomanPrefers(w, m, mCurrent.Value))
            {
                womenMatch[w] = m;
                menMatch[m] = w;
                menMatch[mCurrent.Value] = null;
                if (trace) log.Add($"  -> femme {w} prefere {m} a {mCurrent}, swap");
            }
            else
            {
                if (trace) log.Add($"  -> femme {w} garde {mCurrent}, rejette {m}");
            }
        }

        var matching = new Dictionary<int, int>();
        for (int m = 0; m < n; m++) matching[m] = menMatch[m]!.Value;
        return new GsResult(matching, log);
    }

    // Rendu d'un couplage au meme format que le repr dict du twin Python : {0: 3, 1: 0, ...}
    public static string Show(Dictionary<int, int> mu)
        => "{" + string.Join(", ", mu.OrderBy(kv => kv.Key).Select(kv => $"{kv.Key}: {kv.Value}")) + "}";

    // Egalite de deux couplages, independante de l'ordre d'enumeration.
    public static bool Same(Dictionary<int, int> a, Dictionary<int, int> b)
        => a.Count == b.Count && a.All(kv => b.TryGetValue(kv.Key, out var v) && v == kv.Value);
}

Console.WriteLine("Gs.GaleShapley pret : hommes-proposent, trace optionnelle, au plus n^2 propositions");

Gs.GaleShapley pret : hommes-proposent, trace optionnelle, au plus n^2 propositions


### Interprétation : la boucle `FreeManWithOptions`

Deux conditions d'arrêt pour un homme `m` :

1. `m` est déjà couplé (engagé) → il n'agit pas.
2. `m` a déjà proposé à **toutes** les femmes sans succès → il n'a plus d'option.

Si **tous** les hommes sont dans l'un de ces deux cas, l'algorithme s'arrête. La preuve de terminaison en `n²` propositions utilise le fait qu'à chaque proposition, soit une femme libre devient engagée (incrément du nombre d'engagements), soit une femme engagée change pour un meilleur (le nombre de paires (femme, homme) qu'elle préfère strictement à son actuel **diminue strictement**). Donc au plus `n²` propositions.

En C#, l'état est porté par trois structures locales : `menMatch`/`womenMatch` (`int?[n]`, `null` = libre) et `proposed` (`bool[n, n]`) — l'équivalent exact des dictionnaires et de la matrice du jumeau Python.

## 3. Exemple travaillé : n = 4

Profil tiré de Knuth (1976), exemple 1, page 4 : 4 hommes, 4 femmes, préférences explicites — **identique au jumeau Python**, pour une sortie comparable proposition par proposition.

In [4]:
// === Section 3 : exemple travaille n = 4 -- profil Knuth (1976), exemple 1, page 4 ===
// Identique au twin Python : memes preferences, meme ordre de parcours (homme libre de
// plus petit indice, femme preferee non encore contactee), donc meme trace et meme couplage.
// Hommes 0,1,2,3 -- Femmes 0,1,2,3 ; pref[m][k] = la femme de rang k pour l'homme m.
var menPref4 = new List<List<int>>
{
    new() { 3, 1, 0, 2 },  // m0 : prefere w3 > w1 > w0 > w2
    new() { 0, 3, 2, 1 },  // m1 : prefere w0 > w3 > w2 > w1
    new() { 0, 2, 3, 1 },  // m2 : prefere w0 > w2 > w3 > w1
    new() { 1, 0, 2, 3 },  // m3 : prefere w1 > w0 > w2 > w3
};
var womenPref4 = new List<List<int>>
{
    new() { 0, 1, 2, 3 },  // w0 : prefere m0 > m1 > m2 > m3
    new() { 1, 0, 2, 3 },  // w1 : prefere m1 > m0 > m2 > m3
    new() { 0, 1, 2, 3 },  // w2 : prefere m0 > m1 > m2 > m3
    new() { 2, 3, 0, 1 },  // w3 : prefere m2 > m3 > m0 > m1
};

var prof4 = new PrefProfile(menPref4, womenPref4);
var (matching4, log4) = Gs.GaleShapley(prof4, trace: true);

foreach (var line in log4) Console.WriteLine(line);
Console.WriteLine();
Console.WriteLine("Couplage final : " + Gs.Show(matching4));

Step 1: homme 0 propose a femme 3


  -> femme 3 accepte (libre)


Step 2: homme 1 propose a femme 0


  -> femme 0 accepte (libre)


Step 3: homme 2 propose a femme 0


...

  -> femme 2 accepte (libre)


Step 5: homme 3 propose a femme 1


  -> femme 1 accepte (libre)


Step 5: aucun homme libre restant -- FIN


Couplage final : {0: 3, 1: 0, 2: 2, 3: 1}


### Interprétation : la trace pas-à-pas

L'algorithme a effectué **5 propositions** (au plus `n² = 16`) pour atteindre un couplage parfait stable. À chaque proposition, l'un des trois cas se produit :

1. **Femme libre accepte** : pas de swap, l'engagement passe à 1.
2. **Femme engagée préfère le nouveau** : elle accepte, l'ancien redevient libre. Le nombre d'engagements reste `n`.
3. **Femme engagée garde l'ancien** : le candidat est rejeté (mais ne re-proposera pas à cette femme).

**Le couplage final `{(0,3), (1,0), (2,2), (3,1)}` est-il stable ?** Vérifions dans la section suivante.

## 4. Vérification de la stabilité

Une **paire bloquante** est un couple (homme `m`, femme `w`) tel que `m` préfère strictement `w` à sa partenaire actuelle, **et** `w` préfère strictement `m` à son partenaire actuel. Un couplage est **stable** si et seulement s'il n'admet aucune paire bloquante.

C'est la définition `IsBlockingPair` du lake `Definitions.lean` (ligne 105-108) et `IsStable` (ligne 113-114) :

```lean
def IsBlockingPair (prof : PrefProfile n) (μ : Matching n)
    (m : Fin n) (w : Fin n) : Prop :=
  prof.ManPrefers m w (μ.spouse m) ∧
  prof.WomanPrefers w m (μ.inverse w)

def IsStable (prof : PrefProfile n) (μ : Matching n) : Prop :=
  ∀ m w, ¬ IsBlockingPair prof μ m w
```

In [5]:
// === Section 4 : verification de la stabilite ===
// Miroir de is_blocking_pair / is_stable du twin Python ; traduction directe de
// IsBlockingPair (Definitions.lean:105) et IsStable (Definitions.lean:113).

public static class Stability
{
    // Vrai ssi (m, w) forme une paire bloquante pour le couplage.
    public static bool IsBlockingPair(PrefProfile prof, Dictionary<int, int> matching, int m, int w)
    {
        if (matching[m] == w) return false;  // deja ensemble
        // inverse : quel homme est avec w ?
        int mWithW = matching.First(kv => kv.Value == w).Key;
        // m prefere w a sa partenaire actuelle ?
        bool mPrefers = prof.ManPrefers(m, w, matching[m]);
        // w prefere m a son partenaire actuel ?
        bool wPrefers = prof.WomanPrefers(w, m, mWithW);
        return mPrefers && wPrefers;
    }

    // Vrai ssi le couplage est stable (aucune paire bloquante) -- scan exhaustif O(n^2).
    public static bool IsStable(PrefProfile prof, Dictionary<int, int> matching)
    {
        int n = prof.N;
        for (int m = 0; m < n; m++)
            for (int w = 0; w < n; w++)
                if (IsBlockingPair(prof, matching, m, w)) return false;
        return true;
    }
}

Console.WriteLine("Couplage Gale-Shapley : " + Gs.Show(matching4));
Console.WriteLine($"Stable ? {Stability.IsStable(prof4, matching4)}");

Couplage Gale-Shapley : {0: 3, 1: 0, 2: 2, 3: 1}


Stable ? True


### Interprétation : pourquoi ça marche

La preuve de stabilité (dans le lake, `gale_shapley_stable`, `GaleShapley.lean:104`) procède par contradiction : si `(m, w)` était une paire bloquante, cela signifierait que `m` n'a pas épuisé ses propositions à `w` au moment où l'algorithme s'est arrêté. Mais `m` est soit déjà engagé (donc il préfère sa partenaire actuelle à `w` à la fin, contredisant la première clause de `IsBlockingPair`), soit libre avec aucune option restante (donc `m` a déjà proposé à `w` qui l'a refusé, contredisant la seconde clause de `IsBlockingPair` car `w` n'aurait pas rejeté un homme qu'elle préfère strictement à son actuel).

**C'est l'essence de l'algorithme d'acceptation différée** : les refus sont **définitifs**, et une femme qui rejette un homme aujourd'hui ne le reprendra jamais. Cette asymétrie (propositions multiples côté hommes, acceptation unique côté femmes) est ce qui rend la preuve de stabilité constructive.

## 5. Man-optimalité par énumération exhaustive

Le théorème d'**optimalité homme** (Gale & Shapley 1962) dit que, dans le couplage retourné par l'algorithme, **chaque homme obtient la meilleure partenaire possible parmi tous les couplages stables**. C'est la définition `IsManOptimal` du lake (ligne 120-123) :

```lean
def IsManOptimal (prof : PrefProfile n) (μ : Matching n) : Prop :=
  IsStable prof μ ∧
  ∀ μ' : Matching n, IsStable prof μ' →
    ∀ m : Fin n, prof.menPref m (μ.spouse m) ≤ prof.menPref m (μ'.spouse m)
```

Pour vérifier empiriquement cette propriété sur `n=4`, on **énumère** les `n! = 24` couplages possibles et on identifie ceux qui sont stables. Le couplage Gale-Shapley doit alors avoir la propriété que, pour chaque homme, son rang est **minimal** parmi tous les stables.

In [6]:
// === Section 5 : enumeration exhaustive des n! couplages + verification man-optimalite ===
public static class Enumeration
{
    // Enumere les n! bijections de {0..n-1} vers lui-meme, par echange recursif avec
    // backtracking (chaque permutation est emise exactement une fois).
    // Cout : O(n! * n) -- faisable jusqu'a n ~ 8 (40320 couplages).
    public static List<Dictionary<int, int>> AllMatchings(int n)
    {
        var arr = Enumerable.Range(0, n).ToArray();
        var result = new List<Dictionary<int, int>>();
        void Rec(int i)
        {
            if (i == n)
            {
                var mu = new Dictionary<int, int>();
                for (int m = 0; m < n; m++) mu[m] = arr[m];
                result.Add(mu);
                return;
            }
            for (int j = i; j < n; j++)
            {
                (arr[i], arr[j]) = (arr[j], arr[i]);
                Rec(i + 1);
                (arr[i], arr[j]) = (arr[j], arr[i]);
            }
        }
        Rec(0);
        return result;
    }
}

var allMatch = Enumeration.AllMatchings(prof4.N);
var stableMatch = allMatch.Where(mu => Stability.IsStable(prof4, mu)).ToList();

Console.WriteLine($"Nombre total de couplages : {allMatch.Count}");
Console.WriteLine($"Nombre de couplages stables : {stableMatch.Count}");
Console.WriteLine();
Console.WriteLine("Couplages stables :");
foreach (var mu in stableMatch) Console.WriteLine("  " + Gs.Show(mu));
Console.WriteLine();
Console.WriteLine("Couplage Gale-Shapley : " + Gs.Show(matching4));
Console.WriteLine($"Present dans les stables ? {stableMatch.Any(mu => Gs.Same(mu, matching4))}");

// Verification man-optimalite : pour chaque homme m, son rang dans GS
// doit etre <= son rang dans TOUT autre couplage stable
Console.WriteLine();
Console.WriteLine("Verification man-optimalite :");
bool gsOptimal = true;
for (int m = 0; m < prof4.N; m++)
{
    int rankGs = prof4.MenRank(m, matching4[m]);
    int bestOtherRank = stableMatch.Min(mu => prof4.MenRank(m, mu[m]));
    string status = rankGs <= bestOtherRank ? "OK" : "VIOLATION";
    Console.WriteLine($"  m{m}: rang GS={rankGs}, meilleur autre stable={bestOtherRank} -> {status}");
    if (rankGs > bestOtherRank) gsOptimal = false;
}
Console.WriteLine();
Console.WriteLine($"Le couplage Gale-Shapley est man-optimal : {gsOptimal}");

Nombre total de couplages : 24


Nombre de couplages stables : 2


Couplages stables :


  {0: 1, 1: 0, 2: 2, 3: 3}


  {0: 3, 1: 0, 2: 2, 3: 1}


...

  m0: rang GS=0, meilleur autre stable=0 -> OK


  m1: rang GS=0, meilleur autre stable=0 -> OK


  m2: rang GS=1, meilleur autre stable=1 -> OK


  m3: rang GS=0, meilleur autre stable=0 -> OK


Le couplage Gale-Shapley est man-optimal : True


### Interprétation : la structure du treillis des mariages stables

Pour `n=4` avec ce profil particulier, on a trouvé **2 couplages stables**. Le couplage Gale-Shapley est l'un des deux, et il est **man-optimal** : pour chaque homme, son rang dans GS est meilleur ou égal à son rang dans l'autre couplage stable.

Dualement, GS est **woman-pessimal** : chaque femme obtient sa pire partenaire parmi les stables (vérifiable symétriquement — c'est l'objet de l'exercice 2). C'est l'asymétrie inhérente à la version "hommes proposent" : la partie qui propose a l'avantage, l'autre subit.

**Le lake Lean formalise cette structure via le module `Lattice.lean`** (`exists_isManOptimal`, `exists_isWomanPessimal`) : les couplages stables forment un **distributive lattice** sous l'ordre produit (Knuth 1976, ch. 2 ; Gusfield 1989), et le couplage Gale-Shapley est le **maximum** de ce treillis côté hommes (resp. minimum côté femmes). C'est la structure algébrique qui permet de prouver `gale_shapley_man_optimal` et `gale_shapley_woman_pessimal` conjointement.

## 6. Pont avec la formalisation Lean 4

Le lake [`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/) porte la même théorie en Lean 4, sans aucun `sorry` sur les théorèmes principaux. La table de correspondance C# ↔ Lean :

| Concept C# | Symbole Lean | Fichier | Ligne |
|----------------|--------------|---------|-------|
| `class PrefProfile` | `structure PrefProfile` | `Definitions.lean` | 36 |
| `ManPrefers(m, w1, w2)` | `manPrefers` | `Definitions.lean` | 55 |
| `WomanPrefers(w, m1, m2)` | `womanPrefers` | `Definitions.lean` | 62 |
| `Stability.IsBlockingPair(...)` | `IsBlockingPair` | `Definitions.lean` | 105 |
| `Stability.IsStable(...)` | `IsStable` | `Definitions.lean` | 113 |
| `Gs.GaleShapley(prof)` | `gsGaleShapley prof` | `GSState.lean` | 137 |
| borne `n²` propositions | `gsProposalBound n = n * n` | `GSState.lean` | 135 |
| (vérifié empiriquement) | `gale_shapley_stable` | `GaleShapley.lean` | 104 |
| (vérifié empiriquement) | `gale_shapley_man_optimal` | `GaleShapley.lean` | 130 |
| (déduit) | `gale_shapley_woman_pessimal` | `GaleShapley.lean` | 147 |

**Ce que la simulation C# montre, mais ne prouve pas** : que l'algorithme termine, qu'il retourne un couplage stable, qu'il est man-optimal. La simulation ne teste qu'un **nombre fini** d'instances (`n=4` dans cet exemple). La preuve Lean garantit ces propriétés pour **tout** `n` et **tout** profil de préférences.

**Ce que la preuve Lean ne montre pas** : la *dynamique* de l'algorithme, c'est-à-dire comment les propositions s'enchaînent et pourquoi l'algorithme termine en `n²` étapes (et pas `n³` ou plus). C'est ce que la trace pas-à-pas de la section 3 rend visible.

***

## 7. Exercices

Trois exercices pour s'approprier l'algorithme — les mêmes que le jumeau Python. Les stubs sont conformes à la règle **C.1** (pas d'exception volontaire) : ils affichent leur énoncé et s'exécutent sans erreur, même non complétés.

### Exercice 1 : Gale-Shapley femmes-proposent

**Objectif** : adapter l'algorithme pour que ce soient les **femmes** qui proposent (et les hommes qui acceptent / rejettent).

**Ce que vous devez implémenter** :
- Une fonction `GaleShapleyWomenPropose(prof)` symétrique à `Gs.GaleShapley`.
- Vérifier sur `prof4` que le couplage retourné est **différent** du couplage hommes-proposent.
- Vérifier qu'il est **woman-optimal** (et donc **man-pessimal**).

In [7]:
// Exercice 1 : Gale-Shapley femmes-proposent (stub C.1 -- a completer par l'etudiant)
static GsResult GaleShapleyWomenPropose(PrefProfile prof, bool trace = false)   // TODO etudiant
{
    // A completer :
    // - Initialiser les engagements cote femmes et cote hommes (null = libre)
    // - A chaque etape, trouver une femme libre qui n'a pas propose a tous les hommes
    // - La femme propose a l'homme qu'elle prefere parmi ceux non contactes
    // - L'homme compare et garde sa preferee (swap + liberation de l'ecarte)
    // Indice : miroir exact de Gs.GaleShapley en echangeant les roles homme/femme
    // (WomenPref/WomenRank cote proposeurs, ManPrefers cote acceptants).
    // Le couplage retourne doit etre woman-optimal.
    Console.WriteLine("Exercice 1 a completer par l'etudiant : Gale-Shapley femmes-proposent");
    return new GsResult(new Dictionary<int, int>(), new List<string>());   // TODO etudiant
}

// Test rapide (affiche le couplage vide tant que l'exercice n'est pas complete)
var gsWomen = GaleShapleyWomenPropose(prof4);
Console.WriteLine("Couplage femmes-proposent : " + Gs.Show(gsWomen.Matching));
Console.WriteLine("Couplage hommes-proposent : " + Gs.Show(matching4));
Console.WriteLine($"Identiques ? {Gs.Same(gsWomen.Matching, matching4)}");

Exercice 1 a completer par l'etudiant : Gale-Shapley femmes-proposent


Couplage femmes-proposent : {}


Couplage hommes-proposent : {0: 3, 1: 0, 2: 2, 3: 1}


Identiques ? False


### Exercice 2 : Vérifier woman-pessimalité sur n=4

**Objectif** : montrer empiriquement que le couplage Gale-Shapley (hommes-proposent) est **woman-pessimal** sur le profil `prof4`.

**Ce que vous devez implémenter** :
- Pour chaque couplage stable (énuméré en section 5, variable `stableMatch`), vérifier que pour chaque femme `w`, son rang dans GS est **supérieur ou égal** à son rang dans tout autre couplage stable.
- Afficher le rang de chaque femme dans GS et dans le meilleur-autre-stable.

In [8]:
// Exercice 2 : woman-pessimalite sur n=4 (stub C.1 -- a completer par l'etudiant)
//
// Objectif : montrer que le couplage Gale-Shapley (hommes-proposent) est
//            woman-pessimal sur prof4.
//
// Indice : reutiliser stableMatch (section 5) et matching4.
// Etape 1 : pour chaque femme w, trouver son partenaire dans le couplage GS
//           (matching4) ;
// Etape 2 : comparer son rang GS (prof4.WomenRank) au meilleur rang qu'elle
//           atteint dans un autre couplage stable (stableMatch) ;
// Etape 3 : conclure -- le rang GS doit etre superieur ou egal au meilleur rang
//           des autres stables.
bool? gsPessimal = null;   // TODO etudiant

Console.WriteLine("Exercice 2 : woman-pessimalite de Gale-Shapley");
Console.WriteLine("Resultat : " + (gsPessimal?.ToString() ?? "None"));

Exercice 2 : woman-pessimalite de Gale-Shapley


Resultat : None


### Exercice 3 : Générer un profil aléatoire et vérifier les deux propriétés

**Objectif** : sur 100 profils aléatoires de taille `n=3` ou `n=4`, vérifier que l'algorithme retourne toujours un couplage **stable** et **man-optimal**.

**Ce que vous devez implémenter** :
- Une fonction `RandomProfile(n)` qui génère un profil de préférences aléatoires (permutations uniformes).
- Une boucle sur 100 profils, avec à chaque fois :
  - Calcul du couplage GS
  - Vérification de la stabilité (rapide, O(n²))
  - Vérification de la man-optimalité (lente, O(n! · n) — limiter à `n=3` ou `n=4`)
- Reporter le nombre de violations.

In [9]:
// Exercice 3 : profils aleatoires n=3 x100 (stub C.1 -- a completer par l'etudiant)
static PrefProfile? RandomProfile(int n)   // TODO etudiant
{
    // Objectif : sur 100 profils aleatoires de taille n=3, verifier que l'algorithme
    //            retourne toujours un couplage stable et man-optimal.
    //
    // Indice : une permutation uniforme de {0..n-1} s'obtient par un melange de
    // Fisher-Yates sur Enumerable.Range(0, n).ToArray(), en echangeant chaque
    // position i avec une position tiree dans [i, n) via rng (pose en section 0).
    // Etape 1 : ecrire RandomProfile(n) -> PrefProfile (permutations uniformes
    //           cote hommes et cote femmes) ;
    // Etape 2 : boucler sur 100 profils ; a chaque tirage, verifier la stabilite
    //           (Stability.IsStable) puis la man-optimalite (comparer le rang de
    //           chaque homme a son meilleur rang parmi les couplages stables
    //           enumeres par Enumeration.AllMatchings filtre par IsStable) ;
    // Etape 3 : reporter le nombre de violations de stabilite et d'optimalite.
    Console.WriteLine("Exercice 3 a completer par l'etudiant : verification empirique sur profils aleatoires");
    return null;
}

var profAleatoire = RandomProfile(3);
Console.WriteLine("Exercice 3 : verification empirique sur profils aleatoires");
Console.WriteLine("RandomProfile(3) = " + (profAleatoire?.ToString() ?? "None"));

Exercice 3 a completer par l'etudiant : verification empirique sur profils aleatoires


Exercice 3 : verification empirique sur profils aleatoires


RandomProfile(3) = None



(2,19): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## Conclusion

Ce notebook a présenté l'algorithme de **Gale-Shapley** pour le problème du mariage stable, en twin C# .NET du jumeau Python :

1. **Définitions** : `PrefProfile` (préférences strictes des deux côtés) et `Stability.IsBlockingPair` (paire qui préférerait mutuellement rompre l'engagement).
2. **Algorithme** : hommes-proposent, acceptation différée, terminaison en `n²` propositions.
3. **Stabilité** : vérification empirique sur `n=4` (couplage retourné est stable, 0 paire bloquante).
4. **Man-optimalité** : sur les 2 couplages stables du profil exemple (Knuth 1976), GS donne à chaque homme sa meilleure partenaire possible.
5. **Pont Lean** : les mêmes concepts sont formalisés dans `game_theory_lean/StableMarriage/` (`IsBlockingPair`, `IsStable`, `IsManOptimal`, `gale_shapley_stable`, `gale_shapley_man_optimal`), avec `gsGaleShapley` retournant le couplage et `gsProposalBound n = n*n` la borne de propositions.

**Leçon principale** : la simulation C# permet d'**intuiter** les propriétés de l'algorithme (stabilité, optimalité) sur des instances finies ; la preuve Lean 4 les **garantit** pour tout `n` et tout profil. Les deux se complètent — la simulation pour la dynamique, la preuve pour l'universalité.

**Pour aller plus loin** :

- [Gale & Shapley (1962), College Admissions and the Stability of Marriage](https://www.jstor.org/stable/2312726) — l'article original
- [Knuth (1976), Mariages Stables](https://www-cs-faculty.stanford.edu/~knuth/mariages-stables.html) — introduction canonique en français
- [Roth (2008), Deferred Acceptance Algorithms](https://link.springer.com/article/10.1007/s00182-008-0117-7) — applications modernes (NRMP, 1952-depuis)
- Lake Lean [`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/) — formalisation sans `sorry`
- Jumeau Python [`02-Stable-Marriage-Gale-Shapley.ipynb`](02-Stable-Marriage-Gale-Shapley.ipynb) — le même arc en Python 3

**Retour au sommaire** : [Social Choice](README.md) | [GameTheory](../README.md)

### Références académiques

- Gale, D. & Shapley, L.S. (1962). College Admissions and the Stability of Marriage. **The American Mathematical Monthly** 69(1):9-15.
- Knuth, D.E. (1976). **Mariages Stables et leurs relations avec d'autres problèmes combinatoires**. Les Presses de l'Université de Montréal.
- Roth, A.E. (2008). Deferred Acceptance Algorithms: History, Theory, Practice, and Open Questions. **International Journal of Game Theory** 36(3-4):537-569.
- Gusfield, D. (1989). Three still open problems in the theory of stable marriage. Technical Report CS-89-21, UC Davis.

***

**Navigation** : [<< 01b-Lean-SocialChoice-Formal.ipynb](01b-Lean-SocialChoice-Formal.ipynb) | [03-Voting-Methods-Csharp.ipynb >>](03-Voting-Methods-Csharp.ipynb) | [Index](README.md)